In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load the dataset
df = pd.read_csv('data/ecommerce_algerie_2024_2025_version10mai.csv')
#df = pd.read_csv('data/data.csv')

# Display basic info
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (220000, 14)

Column names:
['Tracking', 'Categorie', 'Produit', 'date expédition', 'date première tentative', 'date livraison/échec', 'date dernier statut', 'ID expediteur', 'depart commune', 'depart wilaya', 'stop desk?', 'destination commune', 'destination wilaya', 'dernier statut']

First few rows:


,Tracking,Categorie,Produit,date expédition,date première tentative,date livraison/échec,date dernier statut,ID expediteur,depart commune,depart wilaya,stop desk?,destination commune,destination wilaya,dernier statut
0,yal-NIFE22,Maison et jardin,Jardin,2024/03/22 15:21,2024/03/23 06:14,2024/03/25 15:44,2024/03/25 19:44,expyal-12877,In Salah,Tamanrasset,oui,Laghouat,Laghouat,Livré
1,yal-ONA71L,Vêtements et accessoires,Bijoux,2024/04/05 17:27,2024/04/07 20:35,2024/04/08 13:36,2024/04/08 15:36,expyal-55462,Alger Centre,Alger,non,M'Chedallah,Bouira,Livré
2,yal-KGZ2XR,Vêtements et accessoires,Vêtements,2025/05/23 16:12,2025/05/24 13:19,2025/05/27 16:40,2025/05/27 17:40,expyal-54033,Jijel,Jijel,oui,Guorriguer,Tébessa,Livré
3,yal-G9SJS2,Appareils électroniques,GPS & navigation,2025/03/08 19:29,2025/03/09 23:09,2025/03/09 23:09,2025/03/09 23:09,expyal-35532,Tizi Ouzou,Tizi Ouzou,non,Alger Centre,Alger,Livré
4,yal-0QCSDQ,Maison et jardin,Sécurité maison,2025/11/20 15:30,2025/11/21 10:11,2025/11/21 10:11,2025/11/21 11:11,expyal-71625,Tébessa,Tébessa,non,Aït Yahia,Tizi Ouzou,Livré


In [3]:
# Define the 4 regions of Algeria (48 wilayas officielles)
regions_mapping = {
    # Centre (10 wilayas)
    'Alger': 'centre',
    'Blida': 'centre',
    'Boumerdès': 'centre',
    'Tipaza': 'centre',
    'Médéa': 'centre',
    'Aïn Defla': 'centre',
    'Bouira': 'centre',
    'Tizi Ouzou': 'centre',
    'Béjaïa': 'centre',
    'Chlef': 'centre',
    "M'Sila": 'centre', 
    
    # Est (14 wilayas)
    'Constantine': 'est',
    'Annaba': 'est',
    'Skikda': 'est',
    'Jijel': 'est',
    'Sétif': 'est',
    'Batna': 'est',
    'Khenchela': 'est',
    'Oum El Bouaghi': 'est',
    'Tébessa': 'est',
    'Guelma': 'est',
    'Souk Ahras': 'est',
    'Mila': 'est',
    'Bordj Bou Arreridj': 'est',
    'El Tarf': 'est',
    
    # Ouest (11 wilayas)
    'Oran': 'ouest',
    'Tlemcen': 'ouest',
    'Sidi Bel Abbès': 'ouest',
    'Mostaganem': 'ouest',
    'Mascara': 'ouest',
    'Relizane': 'ouest',
    'Saïda': 'ouest',
    'Tiaret': 'ouest',
    'Tissemsilt': 'ouest',
    'Aïn Témouchent': 'ouest',
    'Naâma': 'ouest',
    
    # Sud (13 wilayas)
    'Adrar': 'sud',
    'Tamanrasset': 'sud',
    'Illizi': 'sud',
    'Tindouf': 'sud',
    'Béchar': 'sud',
    'El Oued': 'sud',
    'Ouargla': 'sud',
    'Ghardaïa': 'sud',
    'Laghouat': 'sud',
    'Biskra': 'sud',
    'El Bayadh': 'sud',
    'Djelfa': 'sud',
    
}

# Add region column
df['region'] = df['destination wilaya'].map(regions_mapping)

# Check for unmapped wilayas
unmapped = df[df['region'].isna()]['destination wilaya'].unique()
if len(unmapped) > 0:
    print(f"Unmapped wilayas: {unmapped}")
else:
    print("All wilayas are properly mapped to regions")

print(f"\nRegion distribution:\n{df['region'].value_counts()}")

All wilayas are properly mapped to regions

Region distribution:
region
centre    68038
est       66357
ouest     46209
sud       39396
Name: count, dtype: int64


In [19]:
# Always check sparsity before deciding frequency
df["date_exp"] = pd.to_datetime(df["date expédition"], dayfirst=True).dt.date
density_check = (
    df.groupby(["Produit", "region", "date_exp"])
    .size()
    .reset_index(name="order_count")
)
density_check.head()

# How many (product, region) pairs have enough data per frequency?
daily_viable   = density_check.groupby(["Produit","region"])["date_exp"].count()
daily_viable.head()
print(f"Avg daily orders per series: {daily_viable.mean():.1f}")
print(f"Series with >0 daily orders: {(daily_viable > 200).sum()} / {len(daily_viable)}")
density_check.head()


C:\Users\USER\AppData\Local\Temp\ipykernel_14216\1899824713.py:2: UserWarning: Parsing dates in %Y/%m/%d %H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["date_exp"] = pd.to_datetime(df["date expédition"], dayfirst=True).dt.date


Avg daily orders per series: 303.2
Series with >0 daily orders: 348 / 476


,Produit,region,date_exp,order_count
0,Abonnements,centre,2024-01-01,1
1,Abonnements,centre,2024-01-14,1
2,Abonnements,centre,2024-01-25,1
3,Abonnements,centre,2024-01-27,1
4,Abonnements,centre,2024-02-05,1
